# Step by step through the Remora algorithm for signal to sequence alignment

I adapted this from the Remora source code. I changed some variable names and got rid of the parent classes to make it better readable. After executing the code I compare it to the original Remora code directly to confirm that the results are the same.


The following functions are used to calculate the alignments (Structure corresponds to Remora implementation):
```
from_pod5_and_alignment
|
|--add_alignment
|    |
|    |-- trim_signal                
                (trim the signal based on split read tags (sp, ns, ts))
|    |-- revcomp                    
                (calculate the reverse complement if the mapping is reverse)
|    |-- adjust_move_table          
                (Adjust the query to signal alignment in case of reversed signal)
|
|--compute_ref_to_signal
    |
    |--compute_ref_to_signal_inner
        |
        |-- make_sequence_coordinate_mapping    
        |       (Set up the knots used to infer the ref to signal alignment)
        |-- map_ref_to_signal                   
                (Calculate the ref to signal alignment from the knots)
 
```

## Global variables to access some variables inside of functions

In [57]:
import pod5
from remora import io
from pathlib import Path
import numpy as np
test_data_root = Path("../example_data")
pod5_dr = pod5.DatasetReader(test_data_root)
bam_fh = io.ReadIndexedBam(test_data_root / "can_mappings.bam")

read_id = "6e37823a-9398-4be8-b111-65cab029f4e0"
pod5_read = pod5_dr.get_read(read_id)
bam_read = bam_fh.get_first_alignment(read_id)

Indexing BAM by parent read id: 14 Reads [00:00, 14680.06 Reads/s]


## Functions for query- and ref-to-signal alignment

### from_pod5_and_alignment

In [58]:
def from_pod5_and_alignment(pod5_read_record: pod5.reader.ReadRecord, alignment_record, reverse_signal=False):
    
    signal = pod5_read_record.signal
    if reverse_signal:
        signal = signal[::-1]
    # calibration offset and scale apply normalization as:
    #     y=(x+a)/b
    # while io.Read stores shfit and scale as:
    #     y=c*(x-d)
    # In other words, as shift=mean and scale=std. dev.
    alignments = add_alignment(
        signal,
        alignment_record,
        reverse_signal,
    )
    return alignments


### add_alignment

In [59]:
def add_alignment(
    signal,
    alignment_record,
    reverse_signal=False
):
    """Add alignment to read object

    Args:
        alignment_record (pysam.AlignedSegment)
        parse_ref_align (bool): Should reference alignment be parsed
        reverse_signal (bool): Does this read derive from 3' to 5' signal
            (RNA reads)
        pa_scaling (tuple): picoamp to zero-centered picoamp shift and scale
    """
    if (
        alignment_record.reference_name is None
        and alignment_record.is_reverse
    ):
        raise Exception("Unmapped reads cannot map to reverse strand.")

    tags = dict(alignment_record.tags)

    trim_tags = dict(
        (tag, tags.get(tag, dv))
        for tag, dv in (("sp", 0), ("ts", 0), ("ns", None))
    )

    signal = trim_signal(signal, trim_tags, reverse_signal)

    global DEBUG_SIGNAL
    DEBUG_SIGNAL = signal

    parent_read_id = tags.get("pi", None)
    if parent_read_id is None:
        if alignment_record.query_name != read_id:
            raise Exception("Read IDs mismatch")
    else:
        if parent_read_id != read_id:
            raise Exception("Split read IDs mismatch")
        child_read_id = alignment_record.query_name

    seq = alignment_record.query_sequence
    if alignment_record.is_reverse:
        seq = revcomp(seq)
    try:
        stride = tags["mv"][0]
        mv_table = np.array(tags["mv"][1:])
        query_to_signal = np.nonzero(mv_table)[0] * stride
        if signal is not None:
            query_to_signal = adjust_move_table(query_to_signal, 
                                                len(signal), 
                                                len(seq), 
                                                len(mv_table), 
                                                stride, 
                                                reverse_signal=reverse_signal)
    except KeyError:
        raise Exception(f"Move table not found")

    ref_seq = alignment_record.get_reference_sequence().upper()
    cigar = alignment_record.cigartuples
    if alignment_record.is_reverse:
        if ref_seq is not None:
            ref_seq = revcomp(ref_seq)
        cigar = cigar[::-1]
    ref_to_signal = compute_ref_to_signal(query_to_signal, cigar)

    return query_to_signal, ref_to_signal

def trim_signal(signal, trim_tags, reverse_signal):
    if reverse_signal:
        signal = signal[::-1]
    # trim for split read sp tag
    signal = signal[trim_tags["sp"] :]
    # trim for start and end read trimming
    ns = trim_tags["ns"]
    if ns is None:
        ns = signal.size
    signal = signal[trim_tags["ts"] : ns]
    if reverse_signal:
        signal = signal[::-1]

    return signal

def revcomp(seq):
    """Convert seq to its complement sequence and reverse the sequence.
    Handles IUPAC ambiguous bases.
    """
    COMP_BASES = dict(zip(map(ord, "ACGTBVDHKMRY"), map(ord, "TGCAVBHDMKYR")))

    return seq.upper().translate(COMP_BASES)[::-1]

def adjust_move_table(query_to_signal, 
                      sig_len, 
                      seq_len,
                      mv_table_size,
                      stride,
                      reverse_signal=False, 
                      check=True):
    # add last point to query_to_signal
    query_to_signal = np.concatenate(
        [query_to_signal, [sig_len]]
    )
    if reverse_signal:
        query_to_signal = sig_len - query_to_signal[::-1]
    if check:
        if query_to_signal.size - 1 != seq_len:
            raise Exception("Move table discordant with basecalls")
        if mv_table_size != sig_len // stride:
            raise Exception("Move table discordant with signal")
    return query_to_signal


### compute_ref_to_signal

In [60]:
def compute_ref_to_signal(query_to_signal, cigar):
    ref_to_signal = compute_ref_to_signal_inner(
        query_to_signal=query_to_signal,
        cigar=cigar,
    )
    return ref_to_signal

def compute_ref_to_signal_inner(query_to_signal, cigar):
    ref_to_read_knots = make_sequence_coordinate_mapping(cigar)
    return map_ref_to_signal(
        query_to_signal=query_to_signal, ref_to_query_knots=ref_to_read_knots
    )

def make_sequence_coordinate_mapping(cigar):
    """Maps an element in reference to every element in basecalls using
    alignment in `cigar`.

    Args:
        cigar (list): "cigartuples" representing alignment

    Returns:
        array shape (ref_len,). [x_0, x_1, ..., x_(ref_len)]
            such that read_seq[x_i] <> ref_seq[i]. Note that ref_len is derived
            from the cigar input.
    """
    MATCH_OPS = np.array(
        [True, False, False, False, False, False, False, True, True]
    )
    MATCH_OPS_SET = set(np.where(MATCH_OPS)[0])

    QUERY_OPS = np.array([True, True, False, False, True, False, False, True, True])
    REF_OPS = np.array([True, False, True, True, False, False, False, True, True])


    while len(cigar) > 0 and cigar[-1][0] not in MATCH_OPS_SET:
        cigar = cigar[:-1]
    if len(cigar) == 0:
        raise Exception("No match operations found in alignment cigar")
    
    global DEBUG_CIGAR
    DEBUG_CIGAR = cigar

    ops, lens = map(np.array, zip(*cigar))
    if ops.min() < 0 or ops.max() > 8:
        raise Exception("Invalid cigar op(s)")
    if lens.min() < 0:
        raise Exception("Cigar lengths may not be negative")

    is_match = MATCH_OPS[ops]
    match_counts = lens[is_match]
    offsets = np.array([match_counts, np.ones_like(match_counts)])

    # TODO remove knots around ambiguous indels (e.g. left justified HPs)
    # note this requires the ref and query sequences
    ref_knots = np.cumsum(np.where(REF_OPS[ops], lens, 0))
    ref_knots = np.concatenate(
        [[0], (ref_knots[is_match] - offsets).T.flatten(), [ref_knots[-1]]]
    )
    query_knots = np.cumsum(np.where(QUERY_OPS[ops], lens, 0))
    query_knots = np.concatenate(
        [[0], (query_knots[is_match] - offsets).T.flatten(), [query_knots[-1]]]
    )
    knots = np.interp(np.arange(ref_knots[-1] + 1), ref_knots, query_knots)

    global DEBUG_KNOTS, DEBUG_REF_KNOTS, DEBUG_QUERY_KNOTS
    DEBUG_KNOTS = knots
    DEBUG_REF_KNOTS = ref_knots
    DEBUG_QUERY_KNOTS = query_knots

    return knots

def map_ref_to_signal(*, query_to_signal, ref_to_query_knots):
    """Compute interpolated mapping from reference, through query alignment to
    signal coordinates

    Args:
        query_to_signal (np.array): Query to signal coordinate mapping
        ref_to_query_knots (np.array): Reference to query coordinate mapping
    """
    return np.floor(
        np.interp(
            ref_to_query_knots,
            np.arange(query_to_signal.size),
            query_to_signal,
        )
    ).astype(int)



## Perform alignment

In [61]:
query_to_signal, ref_to_signal = from_pod5_and_alignment(pod5_read, bam_read)
print(query_to_signal)
print(ref_to_signal)

[    0    45    50 ... 83205 83225 83234]
[ 1060  1065  1080 ... 82780 82805 82815]


In [62]:
read = io.Read.from_pod5_and_alignment(pod5_read, bam_read)
print(read.query_to_signal)
print(read.ref_to_signal)

[    0    45    50 ... 83205 83225 83234]
[ 1060  1065  1080 ... 82780 82805 82815]


In [63]:
print(query_to_signal == read.query_to_signal, "Number of unequal values:", query_to_signal.size - np.count_nonzero((query_to_signal == read.query_to_signal)))
print(ref_to_signal == read.ref_to_signal, "Number of unequal values:", ref_to_signal.size - np.count_nonzero((ref_to_signal == read.ref_to_signal)))

[ True  True  True ...  True  True  True] Number of unequal values: 0
[ True  True  True ...  True  True  True] Number of unequal values: 0


## Inspecting reference to signal alignment in more detail

Need the following variables for this:
- signal
- query_to_signal
- cigar
- ref_knots
- query_knots
- knots (interpolated)
- ref_to_signal

In [64]:
signal = DEBUG_SIGNAL
cigar = DEBUG_CIGAR
ref_knots = DEBUG_REF_KNOTS
query_knots = DEBUG_QUERY_KNOTS
knots = DEBUG_KNOTS

Comparing the signal from the function to the initial signal: The initial signal is 10 measurements longer. This corresponds to the 

In [85]:
print(pod5_read.signal, len(pod5_read.signal))
print(signal, len(signal))
print(f"Tags:  ts={bam_read.get_tag("ts")}, ns={bam_read.get_tag("ns")} -->", pod5_read.signal[bam_read.get_tag("ts"):bam_read.get_tag("ns")])

[ 976 1002  956 ...  779  745  733] 83244
[ 987  990 1011 ...  779  745  733] 83234
Tags:  ts=10, ns=83244 --> [ 987  990 1011 ...  779  745  733]


Comparing the CIGAR tuples from the function with the original ones: CIGAR tuples are reversed (because read is reverse mapped) and the now last CIGAR element is removed (4=S -> Soft-Clip) (WHY IS IT KEPT IN THE BEGINNING THOUGH?)

In [91]:
print(len(bam_read.cigartuples), bam_read.cigartuples)
print(len(cigar), cigar)
print("Is read reverse?", bam_read.is_reverse)

65 [(4, 36), (0, 655), (2, 1), (0, 23), (2, 1), (0, 664), (2, 1), (0, 517), (1, 1), (0, 3), (1, 1), (0, 166), (2, 4), (0, 107), (2, 1), (0, 169), (1, 6), (0, 206), (2, 1), (0, 39), (2, 2), (0, 4), (1, 1), (0, 241), (2, 1), (0, 4), (2, 2), (0, 502), (1, 2), (0, 122), (1, 1), (0, 2), (2, 1), (0, 59), (2, 1), (0, 60), (2, 1), (0, 13), (2, 1), (0, 19), (1, 1), (0, 168), (2, 2), (0, 33), (1, 2), (0, 486), (2, 1), (0, 496), (1, 3), (0, 462), (1, 1), (0, 31), (2, 1), (0, 14), (1, 2), (0, 518), (1, 1), (0, 2), (1, 1), (0, 7), (1, 1), (0, 910), (2, 1), (0, 54), (4, 86)]
64 [(4, 86), (0, 54), (2, 1), (0, 910), (1, 1), (0, 7), (1, 1), (0, 2), (1, 1), (0, 518), (1, 2), (0, 14), (2, 1), (0, 31), (1, 1), (0, 462), (1, 3), (0, 496), (2, 1), (0, 486), (1, 2), (0, 33), (2, 2), (0, 168), (1, 1), (0, 19), (2, 1), (0, 13), (2, 1), (0, 60), (2, 1), (0, 59), (2, 1), (0, 2), (1, 1), (0, 122), (1, 2), (0, 502), (2, 2), (0, 4), (2, 1), (0, 241), (1, 1), (0, 4), (2, 2), (0, 39), (2, 1), (0, 206), (1, 6), (0, 16

How does the query sequence relate to the query to signal alignment?

In [116]:
print(f"Query (base-called) sequence lenght: {len(bam_read.query_sequence)}, query_to_signal length: {len(query_to_signal)}")

for i in range(10):
    print(f"{bam_read.query_sequence[i]}: [{query_to_signal[i]}, {query_to_signal[i+1]})")

Query (base-called) sequence lenght: 6902, query_to_signal length: 6903
A: [0, 45)
T: [45, 50)
G: [50, 70)
C: [70, 75)
T: [75, 85)
G: [85, 90)
A: [90, 175)
T: [175, 180)
A: [180, 220)
T: [220, 260)


How does the reference sequence relate to the ref to signal alignment?

In [117]:
print(f"Reference sequence lenght: {len(bam_read.query_alignment_sequence)}, query_to_signal length: {len(ref_to_signal)}")

for i in range(10):
    print(f"{bam_read.query_alignment_sequence[i]}: [{ref_to_signal[i]}, {ref_to_signal[i+1]})")

Reference sequence lenght: 6780, query_to_signal length: 6780
T: [1060, 1065)
G: [1065, 1080)
G: [1080, 1090)
C: [1090, 1095)
T: [1095, 1110)
T: [1110, 1120)
T: [1120, 1125)
A: [1125, 1135)
A: [1135, 1155)
C: [1155, 1160)


In [92]:
import plotly.graph_objects as go

In [134]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=signal[:5000]
    )
)


for boundary in query_to_signal[:100]:
    fig.add_vline(x=boundary, line_color="green", line_dash="solid")

for boundary in ref_to_signal[:50]:
    fig.add_vline(x=boundary, line_color="red", line_dash="dot")


fig.show()